# Exercise: Semantic Segmentation with U-Net on Oxford-IIIT Pet Dataset

This notebook demonstrates how to **train, validate, and evaluate a U-Net model**
for **semantic segmentation** using the **Oxford-IIIT Pet Dataset**.

## Learning objectives
- Understand semantic segmentation and pixel-wise classification
- Train a U-Net using Keras / TensorFlow
- Evaluate segmentation performance
- Visualize predictions

## 1. Import dependencies

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import tensorflow_datasets as tfds
from tensorflow.keras.layers import *
from tensorflow.keras import layers
from tensorflow.keras.models import Model

## 2. Load Oxford-IIIT Pet Dataset

In [ ]:
dataset, info = tfds.load('oxford_iiit_pet:4.*.*', with_info=True)
print(info)

## 3. Preprocessing

In [ ]:
IMG_SIZE = 128
NUM_CLASSES = 3  # background, pet, border

In [ ]:
def resize(input_image, input_mask):
    input_image = tf.image.resize(input_image, (IMG_SIZE, IMG_SIZE), method="nearest")
    input_mask = tf.image.resize(input_mask, (IMG_SIZE, IMG_SIZE), method="nearest")
    return input_image, input_mask

def augment(input_image, input_mask):
    if tf.random.uniform(()) > 0.5:
        input_image = tf.image.flip_left_right(input_image)
        input_mask = tf.image.flip_left_right(input_mask)
    return input_image, input_mask

def normalize(input_image, input_mask):
    input_image = tf.cast(input_image, tf.float32) / 255.0
    input_mask -= 1
    return input_image, input_mask

In [ ]:
def load_image_train(datapoint):
    input_image = datapoint["image"]
    input_mask = datapoint["segmentation_mask"]
    input_image, input_mask = resize(input_image, input_mask)
    input_image, input_mask = augment(input_image, input_mask)
    input_image, input_mask = normalize(input_image, input_mask)
    return input_image, input_mask

def load_image_test(datapoint):
    input_image = datapoint["image"]
    input_mask = datapoint["segmentation_mask"]
    input_image, input_mask = resize(input_image, input_mask)
    input_image, input_mask = normalize(input_image, input_mask)
    return input_image, input_mask

In [ ]:
train_dataset = dataset["train"].map(load_image_train, num_parallel_calls=tf.data.AUTOTUNE)
test_dataset = dataset["test"].map(load_image_test, num_parallel_calls=tf.data.AUTOTUNE)
print(train_dataset)

## 4. Creating train and test batches

In [ ]:
BATCH_SIZE = 64
BUFFER_SIZE = 1000

In [ ]:
train_batches = train_dataset.cache().shuffle(BUFFER_SIZE).batch(BATCH_SIZE).repeat()
train_batches = train_batches.prefetch(buffer_size=tf.data.AUTOTUNE)
validation_batches = test_dataset.take(3000).batch(BATCH_SIZE)
test_batches = test_dataset.skip(3000).take(669).batch(BATCH_SIZE)
print(train_batches)

## 5. Visualize dataset

In [ ]:
def display(display_list):
    plt.figure(figsize=(10, 10))
    title = ["Input Image", "True Mask", "Predicted Mask"]
    for i in range(len(display_list)):
        plt.subplot(1, len(display_list), i+1)
        plt.title(title[i])
        plt.imshow(tf.keras.utils.array_to_img(display_list[i]))
        plt.axis("off")
    plt.show()

In [ ]:
sample_batch = next(iter(test_batches))
for _ in range(2):
    random_index = np.random.choice(sample_batch[0].shape[0])
    sample_image, sample_mask = sample_batch[0][random_index], sample_batch[1][random_index]
    display([sample_image, sample_mask])

## 6. Create Simple U-Net Model

In [ ]:
def simple_unet(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES):
    inputs = Input(input_shape)

    c1 = Conv2D(32, 3, activation='relu', padding='same')(inputs)
    p1 = MaxPooling2D()(c1)

    c2 = Conv2D(64, 3, activation='relu', padding='same')(p1)
    p2 = MaxPooling2D()(c2)

    b = Conv2D(128, 3, activation='relu', padding='same')(p2)

    u2 = Conv2DTranspose(64, 2, strides=2, padding='same')(b)
    u2 = Concatenate()([u2, c2])

    u1 = Conv2DTranspose(32, 2, strides=2, padding='same')(u2)
    u1 = Concatenate()([u1, c1])

    outputs = Conv2D(num_classes, 1, activation='softmax')(u1)

    return Model(inputs, outputs)

model_simple = simple_unet()

model_simple.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_simple.summary()

## 7. Create Large U-Net Model

In [ ]:
def double_conv_block(x, n_filters):
    x = layers.Conv2D(n_filters, 3, padding="same", activation="relu", kernel_initializer="he_normal")(x)
    x = layers.Conv2D(n_filters, 3, padding="same", activation="relu", kernel_initializer="he_normal")(x)
    return x

def downsample_block(x, n_filters):
    f = double_conv_block(x, n_filters)
    p = layers.MaxPool2D(2)(f)
    p = layers.Dropout(0.3)(p)
    return f, p

def upsample_block(x, conv_features, n_filters):
    x = layers.Conv2DTranspose(n_filters, 3, 2, padding="same")(x)
    x = layers.concatenate([x, conv_features])
    x = layers.Dropout(0.3)(x)
    x = double_conv_block(x, n_filters)
    return x

In [ ]:
def build_unet_model():
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    # encoder: contracting path
    f1, p1 = downsample_block(inputs, 64)
    f2, p2 = downsample_block(p1, 128)
    f3, p3 = downsample_block(p2, 256)
    f4, p4 = downsample_block(p3, 512)

    # bottleneck
    bottleneck = double_conv_block(p4, 1024)

    # decoder: expanding path
    u6 = upsample_block(bottleneck, f4, 512)
    u7 = upsample_block(u6, f3, 256)
    u8 = upsample_block(u7, f2, 128)
    u9 = upsample_block(u8, f1, 64)

    outputs = layers.Conv2D(3, 1, padding="same", activation="softmax")(u9)

    unet_model = Model(inputs, outputs, name="U-Net")
    return unet_model

In [ ]:
unet_model = build_unet_model()

unet_model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

unet_model.summary()

## 8. Train the U-Net Model (Simple U-Net first)

In [ ]:
NUM_EPOCHS = 20

TRAIN_LENGTH = info.splits["train"].num_examples
STEPS_PER_EPOCH = TRAIN_LENGTH // BATCH_SIZE

VAL_SUBSPLITS = 5
TEST_LENGTH = info.splits["test"].num_examples
VALIDATION_STEPS = TEST_LENGTH // BATCH_SIZE // VAL_SUBSPLITS

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
]

# --- Train Simple U-Net ---
history_simple = model_simple.fit(
    train_batches,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_steps=VALIDATION_STEPS,
    validation_data=validation_batches,
    epochs=NUM_EPOCHS,
    callbacks=callbacks
)

## 9. Plot Training Curves

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_simple.history['loss'], label='Train')
plt.plot(history_simple.history['val_loss'], label='Val')
plt.legend()
plt.title("Loss (Simple U-Net)")

plt.subplot(1, 2, 2)
plt.plot(history_simple.history['accuracy'], label='Train')
plt.plot(history_simple.history['val_accuracy'], label='Val')
plt.legend()
plt.title("Accuracy (Simple U-Net)")

plt.show()

## 10. Visualize Predictions

In [ ]:
def create_mask(pred_mask):
    pred_mask = tf.argmax(pred_mask, axis=-1)
    pred_mask = pred_mask[..., tf.newaxis]
    return pred_mask[0]

def show_predictions(model, dataset=None, num=1):
    if dataset:
        for image, mask in dataset.take(num):
            pred_mask = model.predict(image)
            display([image[0], mask[0], create_mask(pred_mask)])

In [ ]:
show_predictions(model_simple, test_batches.skip(5), 3)

---
# 11. Tasks and Questions (SOLUTIONS)
---

## T11.a) Try with more epochs, are 20 epochs sufficient?

We retrain the simple U-Net with 40 epochs and compare to the 20-epoch run.

In [ ]:
# Rebuild a fresh simple U-Net for fair comparison
model_simple_40 = simple_unet()
model_simple_40.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_simple_40 = model_simple_40.fit(
    train_batches,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_steps=VALIDATION_STEPS,
    validation_data=validation_batches,
    epochs=40,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
    ]
)

In [ ]:
# Compare 20 vs 40 epoch training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_simple.history['val_loss'], label='20 epochs')
axes[0].plot(history_simple_40.history['val_loss'], label='40 epochs')
axes[0].set_title('Validation Loss: 20 vs 40 epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history_simple.history['val_accuracy'], label='20 epochs')
axes[1].plot(history_simple_40.history['val_accuracy'], label='40 epochs')
axes[1].set_title('Validation Accuracy: 20 vs 40 epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"Best val_accuracy at 20 epochs: {max(history_simple.history['val_accuracy']):.4f}")
print(f"Best val_accuracy at 40 epochs: {max(history_simple_40.history['val_accuracy']):.4f}")
print(f"EarlyStopping stopped at epoch: {len(history_simple_40.history['val_loss'])}")

### Answer T11.a)

20 epochs is generally sufficient for the simple U-Net on this dataset. With EarlyStopping 
(patience=5), training typically plateaus around epoch 15-20. When we allow 40 epochs, 
EarlyStopping usually triggers before reaching 40, confirming that additional epochs yield 
diminishing returns. The validation accuracy saturates and may even begin to fluctuate 
slightly as the model starts to overfit. 

For the larger U-Net, more epochs may be beneficial since it has more parameters to 
optimize, but EarlyStopping remains important to prevent overfitting.

---
## T11.b) Study the large U-Net model and describe the differences from the simple U-Net model

In [ ]:
print("=== Simple U-Net ===")
model_simple.summary()
print(f"\nTotal parameters: {model_simple.count_params():,}")

print("\n=== Large U-Net ===")
unet_model.summary()
print(f"\nTotal parameters: {unet_model.count_params():,}")

### Answer T11.b)

| Feature | Simple U-Net | Large U-Net |
|---|---|---|
| **Encoder depth** | 2 levels (32, 64 filters) | 4 levels (64, 128, 256, 512 filters) |
| **Bottleneck** | 128 filters, single Conv2D | 1024 filters, double Conv2D |
| **Conv per level** | 1x Conv2D | 2x Conv2D (double conv block) |
| **Dropout** | None | 0.3 after each pool and upsample |
| **Kernel init** | Default (Glorot) | He Normal |
| **Skip connections** | Yes (2) | Yes (4) |
| **Parameters** | ~100K | ~31M |

Key differences:

1. **Depth**: The large U-Net has 4 downsampling levels vs. 2, allowing it to capture larger-scale context and more abstract features.

2. **Double convolutions**: Each level in the large U-Net applies two consecutive Conv2D layers (the classic U-Net pattern from Ronneberger et al.), giving the network more representational power per level.

3. **Dropout regularization**: The large model uses Dropout(0.3) after every pooling and upsampling step, which helps prevent overfitting given its much larger parameter count.

4. **He Normal initialization**: Better suited for ReLU activations (as described in HOML Ch. 11), avoiding vanishing gradients in deeper networks.

5. **More skip connections**: 4 skip connections vs. 2, preserving more fine-grained spatial detail during upsampling.

---
## T11.c) Select and add a method to measure performance

We add **Mean IoU** (Intersection over Union), the standard metric for semantic segmentation (HOML Ch. 14). 
IoU = TP / (TP + FP + FN) per class, averaged across classes.

In [ ]:
def compute_iou(model, dataset, num_classes=NUM_CLASSES):
    """Compute Mean IoU over a batched dataset."""
    m = tf.keras.metrics.MeanIoU(num_classes=num_classes)
    for images, masks in dataset:
        preds = model.predict(images, verbose=0)
        pred_masks = tf.argmax(preds, axis=-1)  # (batch, H, W)
        true_masks = tf.squeeze(masks, axis=-1)  # (batch, H, W)
        m.update_state(true_masks, pred_masks)
    return m.result().numpy()

# Evaluate Simple U-Net
simple_accuracy = model_simple.evaluate(test_batches, verbose=0)
simple_iou = compute_iou(model_simple, test_batches)

print(f"Simple U-Net - Test Loss: {simple_accuracy[0]:.4f}")
print(f"Simple U-Net - Test Accuracy: {simple_accuracy[1]:.4f}")
print(f"Simple U-Net - Mean IoU: {simple_iou:.4f}")

### Answer T11.c)

We use **Mean Intersection over Union (Mean IoU)**, which is the standard evaluation metric 
for semantic segmentation tasks. Unlike pixel-wise accuracy, which can be misleading when 
classes are imbalanced (e.g., background dominates), IoU measures the overlap between the 
predicted and ground truth regions for each class separately, then averages across classes. 
This gives a fairer picture of how well each class is being segmented.

Formula: IoU = TP / (TP + FP + FN) for each class, then averaged.

---
## T11.d) Compare the performance of the large U-Net with the simple U-Net model

In [ ]:
# Train the large U-Net model
# Note: reduced to 5 epochs for CPU training (34M params is very slow on CPU)
unet_large = build_unet_model()
unet_large.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_large = unet_large.fit(
    train_batches,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_steps=VALIDATION_STEPS,
    validation_data=validation_batches,
    epochs=5,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
    ]
)

In [ ]:
# Evaluate large U-Net
large_accuracy = unet_large.evaluate(test_batches, verbose=0)
large_iou = compute_iou(unet_large, test_batches)

print(f"Large U-Net - Test Loss: {large_accuracy[0]:.4f}")
print(f"Large U-Net - Test Accuracy: {large_accuracy[1]:.4f}")
print(f"Large U-Net - Mean IoU: {large_iou:.4f}")

In [ ]:
# Side-by-side comparison plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_simple.history['val_loss'], label='Simple U-Net')
axes[0].plot(history_large.history['val_loss'], label='Large U-Net')
axes[0].set_title('Validation Loss Comparison')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history_simple.history['val_accuracy'], label='Simple U-Net')
axes[1].plot(history_large.history['val_accuracy'], label='Large U-Net')
axes[1].set_title('Validation Accuracy Comparison')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Summary table
print(f"{'Model':<20} {'Test Accuracy':>15} {'Mean IoU':>12} {'Parameters':>15}")
print("-" * 65)
print(f"{'Simple U-Net':<20} {simple_accuracy[1]:>15.4f} {simple_iou:>12.4f} {model_simple.count_params():>15,}")
print(f"{'Large U-Net':<20} {large_accuracy[1]:>15.4f} {large_iou:>12.4f} {unet_large.count_params():>15,}")

In [ ]:
# Visual comparison of predictions
print("--- Simple U-Net Predictions ---")
show_predictions(model_simple, test_batches.skip(2), 2)

print("--- Large U-Net Predictions ---")
show_predictions(unet_large, test_batches.skip(2), 2)

### Answer T11.d)

The large U-Net is expected to outperform the simple U-Net in both accuracy and Mean IoU, 
particularly on fine-grained boundaries (the "border" class). The deeper encoder captures 
more context, and the additional skip connections preserve more spatial detail. However, 
the large model:
- Is significantly slower to train (~300x more parameters)
- May overfit more easily on this relatively small dataset (~3680 training images)
- Benefits more from Dropout regularization (which is built in)

The simple U-Net still achieves reasonable accuracy and is much faster to iterate with.

---
## T11.e) Compare performance at different input resolutions (Simple U-Net)

In [ ]:
results_resolution = []

for res in [64, 128, 256]:
    print(f"\n{'='*50}")
    print(f"Training Simple U-Net at resolution {res}x{res}")
    print(f"{'='*50}")

    # Rebuild preprocessing for this resolution
    def make_resize_fn(target_size):
        def resize_fn(input_image, input_mask):
            input_image = tf.image.resize(input_image, (target_size, target_size), method="nearest")
            input_mask = tf.image.resize(input_mask, (target_size, target_size), method="nearest")
            return input_image, input_mask
        return resize_fn

    resize_fn = make_resize_fn(res)

    def load_train_res(datapoint):
        img = datapoint["image"]
        mask = datapoint["segmentation_mask"]
        img, mask = resize_fn(img, mask)
        img, mask = augment(img, mask)
        img, mask = normalize(img, mask)
        return img, mask

    def load_test_res(datapoint):
        img = datapoint["image"]
        mask = datapoint["segmentation_mask"]
        img, mask = resize_fn(img, mask)
        img, mask = normalize(img, mask)
        return img, mask

    train_ds = dataset["train"].map(load_train_res, num_parallel_calls=tf.data.AUTOTUNE)
    test_ds = dataset["test"].map(load_test_res, num_parallel_calls=tf.data.AUTOTUNE)

    train_b = train_ds.cache().shuffle(BUFFER_SIZE).batch(BATCH_SIZE).repeat()
    train_b = train_b.prefetch(buffer_size=tf.data.AUTOTUNE)
    test_b = test_ds.skip(3000).take(669).batch(BATCH_SIZE)

    # Build model at this resolution
    model_res = simple_unet(input_shape=(res, res, 3), num_classes=NUM_CLASSES)
    model_res.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    hist = model_res.fit(
        train_b,
        steps_per_epoch=STEPS_PER_EPOCH,
        validation_steps=VALIDATION_STEPS,
        validation_data=test_ds.take(3000).batch(BATCH_SIZE),
        epochs=20,
        callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
        verbose=1
    )

    test_eval = model_res.evaluate(test_b, verbose=0)
    iou = compute_iou(model_res, test_b)

    results_resolution.append({
        'resolution': res,
        'test_loss': test_eval[0],
        'test_accuracy': test_eval[1],
        'mean_iou': iou,
        'params': model_res.count_params()
    })

    print(f"Resolution {res}: Acc={test_eval[1]:.4f}, IoU={iou:.4f}")

In [ ]:
# Plot resolution comparison
resolutions = [r['resolution'] for r in results_resolution]
accuracies = [r['test_accuracy'] for r in results_resolution]
ious = [r['mean_iou'] for r in results_resolution]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar([str(r) for r in resolutions], accuracies, color=['#2196F3', '#4CAF50', '#FF9800'])
axes[0].set_title('Test Accuracy vs Resolution')
axes[0].set_xlabel('Resolution')
axes[0].set_ylabel('Accuracy')
axes[0].grid(True, axis='y')

axes[1].bar([str(r) for r in resolutions], ious, color=['#2196F3', '#4CAF50', '#FF9800'])
axes[1].set_title('Mean IoU vs Resolution')
axes[1].set_xlabel('Resolution')
axes[1].set_ylabel('Mean IoU')
axes[1].grid(True, axis='y')

plt.tight_layout()
plt.show()

print(f"\n{'Resolution':<12} {'Accuracy':>12} {'Mean IoU':>12} {'Parameters':>12}")
print("-" * 50)
for r in results_resolution:
    print(f"{r['resolution']:<12} {r['test_accuracy']:>12.4f} {r['mean_iou']:>12.4f} {r['params']:>12,}")

### Answer T11.e)

Higher input resolution generally improves segmentation quality:

- **64x64**: Lowest accuracy/IoU. Many fine details (ears, tails, borders) are lost during 
  downscaling. The model has fewer spatial pixels to classify, making boundary regions 
  particularly hard to segment.

- **128x128** (default): Good balance between quality and training speed. Captures most 
  structural features of the pets.

- **256x256**: Best segmentation quality, especially on borders. More spatial detail is 
  preserved, but training is ~4x slower per epoch and memory usage increases significantly.

The parameter count stays the same regardless of resolution (convolutional weights are 
resolution-independent), but the intermediate feature maps grow quadratically with 
resolution, increasing memory and compute cost.

---
## Questions (ANSWERS)
---

### Q11.a) How does segmentation differ from classification?

**Classification** assigns a single label to an entire image (e.g., "this image contains a 
cat"). The output is a vector of class probabilities of shape `(num_classes,)`.

**Semantic segmentation** assigns a class label to every individual pixel in the image. The 
output has the same spatial dimensions as the input: `(H, W, num_classes)`. Each pixel gets 
its own classification.

As discussed in HOML Chapter 14, segmentation can be seen as a dense prediction task: 
instead of collapsing spatial information into a single prediction (as a classifier does 
with global average pooling or flattening), a segmentation network must preserve and 
reconstruct spatial resolution. This is why architectures like FCN and U-Net use an 
encoder-decoder structure with upsampling layers.

| Aspect | Classification | Segmentation |
|---|---|---|
| Output | One label per image | One label per pixel |
| Output shape | (num_classes,) | (H, W, num_classes) |
| Architecture | Encoder + FC head | Encoder + Decoder |
| Loss function | Categorical crossentropy | Sparse categorical crossentropy (per-pixel) |

### Q11.b) What data augmentation is used and where?

The augmentation used is **random horizontal flipping**, applied in the `augment()` function. 
With 50% probability, both the image and its corresponding segmentation mask are flipped 
left-to-right simultaneously.

```python
def augment(input_image, input_mask):
    if tf.random.uniform(()) > 0.5:
        input_image = tf.image.flip_left_right(input_image)
        input_mask = tf.image.flip_left_right(input_mask)
    return input_image, input_mask
```

This is applied **only to the training set** (inside `load_image_train()`), not to the test 
set (`load_image_test()` skips the `augment()` call). This is correct practice: augmentation 
increases training set diversity to reduce overfitting, but test data should remain 
unmodified for consistent evaluation.

It is critical that the same geometric transformation is applied to both the image and its 
mask simultaneously. Otherwise the pixel-level labels would no longer correspond to the 
correct image regions.

### Q11.c) What is the purpose of skip connections?

Skip connections (also called shortcut connections) in U-Net **concatenate feature maps from 
the encoder directly to the corresponding decoder level**. Their purpose is to recover 
fine-grained spatial information that is lost during downsampling.

During encoding (downsampling), the network learns increasingly abstract features but loses 
spatial resolution. During decoding (upsampling), the network must reconstruct full-resolution 
output. Without skip connections, the decoder would have to reconstruct spatial detail purely 
from the compressed bottleneck representation, which is extremely difficult.

By concatenating encoder feature maps to the decoder at each level:

1. **Spatial detail is preserved**: High-resolution features (edges, textures) from early 
   encoder layers help the decoder produce sharp, accurate boundaries.

2. **Gradient flow is improved**: Skip connections provide shorter paths for gradients during 
   backpropagation, making training more stable (similar to ResNet skip connections, HOML Ch. 14).

3. **Multi-scale feature fusion**: The decoder combines low-level spatial features (from 
   encoder) with high-level semantic features (from bottleneck), enabling accurate 
   pixel-level classification.

In the simple U-Net:
```python
u2 = Concatenate()([u2, c2])  # skip from encoder level 2
u1 = Concatenate()([u1, c1])  # skip from encoder level 1
```

---
### Reflection

This exercise demonstrated the complete pipeline for semantic segmentation using U-Net, 
following the approach from HOML Chapter 14. Key takeaways:

- The encoder-decoder architecture with skip connections is the foundation of modern 
  segmentation networks (FCN, U-Net).
- Mean IoU is a better evaluation metric than pixel accuracy for segmentation, since it 
  accounts for class imbalance.
- Model capacity (simple vs. large) and input resolution both significantly impact 
  segmentation quality, but with diminishing returns and increased computational cost.
- EarlyStopping is essential to prevent overfitting, especially with larger models.
- Data augmentation (even simple horizontal flipping) helps generalization on small datasets.